# Day 12 – RAG Basics – Extensive Solutions

Complete solutions for all exercises with detailed explanations and optimisations.

## Exercise 1: PDF loader

Load a PDF, split it, and build a RAG chain.

In [ ]:
# !pip install pypdf langchain chromadb openai tiktoken

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
import os

# Load PDF
pdf_path = "sample.pdf"  # replace with your file
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

# Embed and store
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# QA chain
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True)

# Ask a question
question = "What is the main topic of this document?"
result = qa({"query": question})
print(f"Answer: {result['result']}")
print(f"Sources: {[doc.metadata.get('source') for doc in result['source_documents']]}")

## Exercise 2: Change chunk size and compare

We'll test chunk sizes 200, 500, 1000 and evaluate answer quality manually.

In [ ]:
chunk_sizes = [200, 500, 1000]
test_question = "What is the capital of France?"  # ensure your PDF contains this

for cs in chunk_sizes:
    print(f"\n--- Chunk size = {cs} ---")
    splitter = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=cs//10)
    chunks = splitter.split_documents(documents)
    vectorstore = Chroma.from_documents(chunks, embeddings)
    qa = RetrievalQA.from_chain_type(llm, retriever=vectorstore.as_retriever())
    answer = qa.run(test_question)
    print(f"Answer: {answer[:200]}...")
    # Note: smaller chunks may miss context, larger chunks may include irrelevant text.

## Exercise 3: Add memory (conversational RAG)

Use `ConversationalRetrievalChain` to maintain chat history.

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    memory=memory
)

# Simulate a conversation
print(conversation_chain.run("What is the Eiffel Tower?"))
print(conversation_chain.run("How tall is it?"))  # understands 'it' refers to Eiffel Tower
print(conversation_chain.run("Where is it located?"))

# Inspect memory
print("\nChat history:")
for msg in memory.chat_memory.messages:
    print(f"{msg.type}: {msg.content[:50]}...")

## Exercise 4: Switch to local embeddings

Use `HuggingFaceEmbeddings` instead of OpenAI to avoid API costs.

In [ ]:
# !pip install sentence-transformers
from langchain.embeddings import HuggingFaceEmbeddings

local_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore_local = Chroma.from_documents(chunks, local_embeddings, persist_directory="./chroma_local")
qa_local = RetrievalQA.from_chain_type(llm, retriever=vectorstore_local.as_retriever())

answer = qa_local.run("What is the main topic?")
print(answer)

# Note: local embeddings are slower but free and private.

## Bonus: Persistent vector store

Save the vector store to disk and reload later without recomputing embeddings.

In [ ]:
# Save
vectorstore.persist()  # if using Chroma with persist_directory

# Load later
vectorstore_loaded = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
retriever_loaded = vectorstore_loaded.as_retriever()
print("Vector store reloaded successfully")